# 🌾 Atelier IA Agricole — 02. VLM : comprendre des images de plantes

Un **VLM** (*Vision-Language Model*) comprend à la fois les **images** et le **texte**.
On peut lui montrer une **photo de feuille** et lui demander de la **décrire** ou de
répondre à une **question** à son sujet.

Ce notebook n'est pas un cours de programmation : le code reste minimal. L'objectif est
d'apprendre à **utiliser** un modèle de fondation vision-langage sur un vrai jeu de données,
et d'observer l'effet de réglages comme la **longueur de réponse** et le **prompt engineering**
(la façon de poser la question).

On utilise **SmolVLM** (Hugging Face), un VLM **compact (~250 à 500 M de paramètres)**, assez
léger pour tourner sur le CPU gratuit de Google Colab.


In [ ]:
# === Configuration de l'environnement (exécuter en premier) ===
import os, sys, subprocess

# Sommes-nous dans Google Colab ?
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

# MODE_DEMO = True  -> utilise de tout petits modèles / jeux de données réduits
# (utile pour tester rapidement, ou hors GPU). Mettez False pour l'atelier complet.
# La variable d'environnement ATELIER_DEMO permet de forcer le mode pour les tests.
MODE_DEMO = os.environ.get("ATELIER_DEMO", "0") == "1"

def pip_install(*paquets):
    """Installe des paquets pip de façon silencieuse (Colab ou local)."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *paquets]
    print("Installation :", " ".join(paquets), "...")
    subprocess.run(cmd, check=False)

print(f"IS_COLAB = {IS_COLAB}")
print(f"MODE_DEMO = {MODE_DEMO}")
print("Python :", sys.version.split()[0])


In [ ]:
pip_install("transformers>=4.56", "accelerate", "pillow")
from transformers import pipeline
print("✅ Bibliothèques prêtes.")


In [ ]:
pip_install("kagglehub", "pandas")
import kagglehub

def telecharger_dataset_kaggle(reference):
    """Télécharge un jeu de données Kaggle public et renvoie son dossier local.
    Renvoie None si Kaggle est injoignable (un échantillon de secours prend alors le relais)."""
    try:
        dossier = kagglehub.dataset_download(reference)
        print(f"✅ Jeu de données Kaggle prêt : {reference}")
        return dossier
    except Exception as e:
        print(f"⚠️ Kaggle indisponible ({e}) → utilisation d'un échantillon de secours.")
        return None


In [ ]:
pip_install("pillow")
import os, random, io, urllib.request
from PIL import Image

REFERENCE_IMAGES_KAGGLE = "rashikrahmanpritom/plant-disease-recognition-dataset"
URLS_SECOURS = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/1/16/Leaf_curl_on_peach.jpg/960px-Leaf_curl_on_peach.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f6/Jowar_leaf_infested_by_pest_or_disease.jpg/960px-Jowar_leaf_infested_by_pest_or_disease.jpg",
]

def echantillon_images_plantes(n=10):
    """Renvoie une liste de (image PIL, label) : n photos du jeu Kaggle Healthy/Powdery/Rust ;
    à défaut, quelques URLs publiques ; et en dernier recours une image synthétique hors-ligne.
    Ainsi la liste renvoyée n'est jamais vide, même sans réseau."""
    dossier = telecharger_dataset_kaggle(REFERENCE_IMAGES_KAGGLE)
    if dossier:
        try:
            base = os.path.join(dossier, "Test", "Test")
            fichiers = []
            for classe in sorted(os.listdir(base)):
                dossier_c = os.path.join(base, classe)
                if not os.path.isdir(dossier_c):
                    continue
                for nom in sorted(os.listdir(dossier_c))[:20]:
                    fichiers.append((os.path.join(dossier_c, nom), classe))
            random.Random(42).shuffle(fichiers)
            images = [(Image.open(chemin).convert("RGB"), label) for chemin, label in fichiers[:n]]
            if images:
                return images
            print("⚠️ Jeu Kaggle vide/inattendu → images de secours.")
        except Exception as e:
            print(f"⚠️ Lecture du jeu Kaggle impossible ({e}) → images de secours.")

    entetes = {"User-Agent": "Mozilla/5.0 (AtelierIA-Agricole; educatif)"}
    images = []
    for url in URLS_SECOURS[:n]:
        try:
            req = urllib.request.Request(url, headers=entetes)
            with urllib.request.urlopen(req, timeout=20) as r:
                images.append((Image.open(io.BytesIO(r.read())).convert("RGB"), "inconnu"))
        except Exception as e:
            print("⚠️ Image ignorée :", e)

    # Dernier recours 100% hors-ligne : si Kaggle ET les URLs échouent, on fabrique au moins
    # une image synthétique pour que le notebook aille jusqu'au bout (pas d'IndexError sur
    # echantillon[0], liste jamais vide).
    if not images:
        from PIL import ImageDraw
        print("⚠️ Aucune image en ligne → image de secours générée localement.")
        for _ in range(max(1, min(n, 3))):
            img = Image.new("RGB", (256, 256), (150, 170, 90))
            ImageDraw.Draw(img).ellipse([40, 40, 216, 216], fill=(70, 110, 50))
            images.append((img, "inconnu"))
    return images


## 1. Charger le VLM

- En **mode démo** : `SmolVLM-256M-Instruct` (250M paramètres, rapide à tester).
- En **mode complet** : `SmolVLM-500M-Instruct` (500M paramètres, plus précis).

On désactive le **découpage en tuiles** de l'image (`do_image_splitting`) : cela suffit pour
décrire une photo simple, et ça reste **rapide** même sans GPU.


In [ ]:
MODELE_VLM = "HuggingFaceTB/SmolVLM-256M-Instruct" if MODE_DEMO else "HuggingFaceTB/SmolVLM-500M-Instruct"

pipe = pipeline("image-text-to-text", model=MODELE_VLM, dtype="auto")
pipe.image_processor.do_image_splitting = False

n_params = pipe.model.num_parameters() / 1e6
print(f"✅ {MODELE_VLM} chargé (~{n_params:.0f} M paramètres)")


## 2. Décrire et interroger une image

Une seule fonction suffit : on donne une **image** et une **question** (texte), le modèle
répond. `max_new_tokens` limite la longueur de la réponse générée.


In [ ]:
def demander_image(image, question="Describe the image in one short sentence.", max_new_tokens=40):
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                              {"type": "text", "text": question}]}]
    sortie = pipe(text=messages, max_new_tokens=max_new_tokens)
    return sortie[0]["generated_text"][-1]["content"].strip()


## 3. Un vrai jeu de données agricole (Kaggle)

On utilise le jeu **Plant Disease Recognition** (Kaggle) : des photos de feuilles réparties
en 3 catégories — `Healthy`, `Powdery`, `Rust`. On demande au VLM un **diagnostic rapide** sur
plusieurs photos, et on compare à la vraie catégorie.


In [ ]:
N_EXEMPLES = 3 if MODE_DEMO else 10
echantillon = echantillon_images_plantes(N_EXEMPLES)
print(f"{len(echantillon)} images chargées.")

question_diagnostic = ("Look at this plant leaf. In one short sentence, say if it looks "
                       "healthy or diseased, and why.")

for image, vrai_label in echantillon:
    reponse = demander_image(image, question_diagnostic)
    print(f"Vrai : {vrai_label:10s} | VLM : {reponse}")


> 🌍 **Remarque langue.** Ces petits modèles VLM « pensent » surtout en **anglais**.
> Astuce : posez la question en anglais, puis **traduisez la réponse** avec un SLM/LLM
> (notebooks 01 et 05) !


---
# 🏋️ Exercices

Ces exercices ne visent pas à *écrire du code*, mais à **observer l'effet** de quelques
réglages sur un modèle vision-langage.


### 🏋️ Exercice 1 — Effet de `max_new_tokens`

Prenez la première image de `echantillon` et demandez une description avec
`max_new_tokens=10`, puis `max_new_tokens=80`. Que se passe-t-il avec la réponse courte ?


In [ ]:
# 👉 Votre code ici


### ✅ Solution 1


In [ ]:
image_test, _ = echantillon[0]
for taille in [10, 80]:
    print(f"--- max_new_tokens = {taille} ---")
    print(demander_image(image_test, "Describe this leaf in detail.", max_new_tokens=taille))
    print()


### 🏋️ Exercice 2 — Prompt engineering : question vague vs question ciblée

Sur la même image, comparez une question **vague** (« What do you see? ») à une question
**ciblée** (« Does this leaf show signs of rust disease, powdery mildew, or is it healthy?
Answer with one word. »). Laquelle donne une réponse plus utile pour un diagnostic ?


In [ ]:
# 👉 Votre code ici


### ✅ Solution 2


In [ ]:
prompt_vague = "What do you see?"
prompt_cible = ("Does this leaf show signs of rust disease, powdery mildew, or is it "
                "healthy? Answer with one word.")

print("=== Vague ===")
print(demander_image(image_test, prompt_vague))
print("\n=== Ciblée ===")
print(demander_image(image_test, prompt_cible))


## ✅ Récapitulatif

- Un **VLM** relie **image** et **texte** : description ou question/réponse visuelle.
- `max_new_tokens` contrôle la longueur ; une question **ciblée** (prompt engineering) donne
  une réponse plus exploitable qu'une question vague.
- Très utile pour le **pré-diagnostic** à partir d'une photo — à confirmer toujours par un
  expert.

**➡️ Notebook suivant : `03_TinyLLM_TinyVLM.ipynb`** — les modèles génératifs les plus petits.
